# CHSM8, full pipeline

Every stage of the real run, with our exact commands (`recipes/`, which call the same `src/` scripts as `nano.ipynb`).
GPU stages assume GH200-class GPUs; times are ours. Each cell is one recipe; run them in order, or run the recipes from a shell.
Set `NPROC` / `NGPU` for your machine; multi-node: replace `torchrun --standalone` with your launcher.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

## 0. Data (CPU)
Hugging Face → the formats we trained on: packed pretraining records (`data/3p`, about 1.5 TB), the SFT/DPO corpus
(`data/sft30m_v21`), the games-only corpus (`data/games30m`), the 14 player packs (`data/players/<key>`) and player-prompt
corpus (`data/sft_players`), and the labeler's distillation set (`data/labeler`).

In [ ]:
!bash recipes/0_data.sh

## 1. Pretraining: chsm8-pt
78.3B positions, 4 GPUs, about 40 hours (three 12-hour legs with `--resume`). Result: 1837 Elo.

In [ ]:
!bash recipes/1_pretrain.sh

## 2. Games-only continuation: chsm8-player-base
26.0M post-training games, no text, 4 GPUs, about 65 minutes. Result: 1890 Elo.

In [ ]:
!bash recipes/2_player_base.sh

## 3. Player models
14 fine-tunes on one player's own moves, 1 GPU, minutes each. Elo 1850–1910.

In [ ]:
!bash recipes/3_players.sh

## 4. Teacher labels (optional)
Gemma-4-31B describing game sides. Our labels are public ([chsm8-teacher-labels](https://huggingface.co/datasets/LegumMagister/chsm8-teacher-labels),
[chsm8-distillation](https://huggingface.co/datasets/LegumMagister/chsm8-distillation)); run this only to label new games.
It needs an OpenAI-compatible Gemma-4-31B endpoint (see the recipe).

In [ ]:
# !bash recipes/4_teacher.sh

## 5. Labeler: chsm8-labeler-gemma3-1b
Gemma-3-1B full fine-tune on the 2.40M clean teacher sets, 1 epoch, 4 GPUs, about 5 hours.

In [ ]:
!bash recipes/5_labeler.sh

## 6. Labeling (optional)
The labeler describes game sides with vLLM, one worker per GPU. The released corpus (chsm8-post-training) is already labeled.

In [ ]:
# !bash recipes/6_label.sh

## 7. SFT: chsm8-sft
One pass over 27.1M described sides + player prompts, 8 GPUs, 3.1 hours. Result: 1792 Elo without a prompt.

In [ ]:
!bash recipes/7_sft.sh

## 8. DPO: chsm8
20 minutes on 1 GPU. Result: 1797 Elo without a prompt; style adherence up on every prompt.

In [ ]:
!bash recipes/8_dpo.sh

## 9. Elo
1,000 games against Stockfish 17.1 at `UCI_Elo` 2000, 0.05 s per move. `STOCKFISH` = your binary.

In [ ]:
!bash recipes/9_eval.sh games checkpoints/pt100/chsm8_latest.pt pt
!bash recipes/9_eval.sh games checkpoints/games/g100_drop_lr5e-05/latest.pt player_base
!bash recipes/9_eval.sh text checkpoints/sft_v21/sft_v21.pt sft
!bash recipes/9_eval.sh text checkpoints/dpo_v21/v21_label_kl1.pt dpo
!bash recipes/9_eval.sh text checkpoints/dpo_v21/v21_label_kl1.pt dpo_queens "trade queens as early as possible"